## Import libraries and set up paths

In [ ]:
import rasterio
from rasterio.features import shapes
import geopandas as gpd
import pandas as pd
import numpy as np
import os
from pathlib import Path
from shapely.geometry import shape

# Workspace-relative paths
REPO_ROOT = Path.cwd().resolve()
if (REPO_ROOT / 'notebooks').exists():
    REPO_ROOT = REPO_ROOT.parent
DATA_DIR = REPO_ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
RESULTS_DIR = REPO_ROOT / 'results'

# Set workspace paths
WORKSPACE = str(REPO_ROOT)
RECLASS_FOLDER = os.fspath(RESULTS_DIR / 'tsc' / 'reclass')
OUTPUT_FOLDER = os.fspath(RESULTS_DIR / 'tsc' / 'overlay_output')
CLUSTERS_SHP = os.fspath(RAW_DIR / 'neighborhood_shapefile' / 'Nabolag_cph_fre_new.shp')
WEIGHTS_EXCEL = os.fspath(RAW_DIR / '0045AHP_Vægtninger.xlsx')

# Create output folders
os.makedirs(RECLASS_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print(f"Workspace: {WORKSPACE}")
print(f"Reclassified rasters: {RECLASS_FOLDER}")
print(f"Output folder: {OUTPUT_FOLDER}")
print(f"\nChecking if files exist:")
print(f"  Clusters shapefile: {os.path.exists(CLUSTERS_SHP)}")
print(f"  Weights Excel: {os.path.exists(WEIGHTS_EXCEL)}")

Workspace: /Users/jacobsmacbookpro/P7_pyt
Reclassified rasters: /Users/jacobsmacbookpro/P7_pyt/0022tsc_reclass
Output folder: /Users/jacobsmacbookpro/P7_pyt/0041overlay_output

Checking if files exist:
  Clusters shapefile: True
  Weights Excel: True


## Load weighting from Excel

In [13]:
# Read weights from Excel using openpyxl directly (skip DataFrame for now)
import openpyxl

wb = openpyxl.load_workbook(WEIGHTS_EXCEL)
ws = wb.active

# Read as simple lists
weight_data = []
for row in ws.iter_rows(min_row=2, values_only=True):
    if row[0] is not None:
        weight_data.append((str(row[0]), float(row[1])))

print("✓ Successfully read Excel file")
print(f"\nWeights loaded: {len(weight_data)} variables")
print("\nWeights table:")
for var, weight in weight_data:
    print(f"  {var:30s}: {weight:.2f}")

# Store as dictionary for later use
weights_dict_raw = {var: weight for var, weight in weight_data}
print(f"\nVariables: {list(weights_dict_raw.keys())}")

✓ Successfully read Excel file

Weights loaded: 20 variables

Weights table:
  mean_price                    : 8.78
  mean_sqm                      : 8.88
  public_housing                : 9.72
  mig_in                        : 5.80
  mig_out                       : 7.12
  PMB                           : 2.07
  PUB                           : 2.20
  EMUB                          : 1.42
  age_18_25                     : 1.73
  age_26_40                     : 3.73
  age_41_55                     : 2.23
  age_56_69                     : 1.50
  crime_main_y                  : 2.35
  grund                         : 3.47
  gym_erhv                      : 2.70
  lvu                           : 6.38
  disp_inc                      : 12.05
  emp                           : 6.17
  unemp                         : 6.78
  ool                           : 4.95

Variables: ['mean_price', 'mean_sqm', 'public_housing', 'mig_in', 'mig_out', 'PMB', 'PUB', 'EMUB', 'age_18_25', 'age_26_40', 'age_41_55', 'ag

## Excluded layers
We exclude: qol, mig_net, counts

In [14]:
# Excluded layers
excluded = ['qol', 'mig_net', 'counts']

# Filter weights (remove excluded variables)
weight_dict = {k: v for k, v in weights_dict_raw.items() if k not in excluded}

print(f"Original variables: {len(weights_dict_raw)}")
print(f"After excluding {excluded}: {len(weight_dict)}")
print(f"\nRemaining variables and weights:")
for var, weight in sorted(weight_dict.items(), key=lambda x: x[1], reverse=True):
    print(f"  {var:30s}: {weight:.2f}")

# Normalize weights so they sum to 1
total_weight = sum(weight_dict.values())
normalized_weights = {k: v/total_weight for k, v in weight_dict.items()}

print(f"\nTotal weight before normalization: {total_weight:.2f}")
print(f"Normalized weights (sum to 1):")
for var, w in sorted(normalized_weights.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {var:30s}: {w:.4f}")
print(f"  ...")
print(f"Sum of normalized weights: {sum(normalized_weights.values()):.4f}")

Original variables: 20
After excluding ['qol', 'mig_net', 'counts']: 20

Remaining variables and weights:
  disp_inc                      : 12.05
  public_housing                : 9.72
  mean_sqm                      : 8.88
  mean_price                    : 8.78
  mig_out                       : 7.12
  unemp                         : 6.78
  lvu                           : 6.38
  emp                           : 6.17
  mig_in                        : 5.80
  ool                           : 4.95
  age_26_40                     : 3.73
  grund                         : 3.47
  gym_erhv                      : 2.70
  crime_main_y                  : 2.35
  age_41_55                     : 2.23
  PUB                           : 2.20
  PMB                           : 2.07
  age_18_25                     : 1.73
  age_56_69                     : 1.50
  EMUB                          : 1.42

Total weight before normalization: 100.03
Normalized weights (sum to 1):
  disp_inc                      : 0.120

## Find all reclassified rasters

In [15]:
# Find all reclassified rasters
raster_files = [f for f in os.listdir(RECLASS_FOLDER) if f.endswith('.tif')]
raster_files = sorted(raster_files)

print(f"Found {len(raster_files)} rasters:")
for raster in raster_files:
    print(f"  - {raster}")

# Extract variable names from raster filenames
raster_dict = {}
for raster in raster_files:
    var_name = raster.replace('reclass_', '').replace('.tif', '')
    raster_dict[var_name] = os.path.join(RECLASS_FOLDER, raster)

print(f"\nRaster dictionary keys: {list(raster_dict.keys())}")

Found 23 rasters:
  - reclass_EMUB.tif
  - reclass_PMB.tif
  - reclass_PUB.tif
  - reclass_age_18_25.tif
  - reclass_age_26_40.tif
  - reclass_age_41_55.tif
  - reclass_age_56_69.tif
  - reclass_counts.tif
  - reclass_crime_main_y.tif
  - reclass_disp_inc.tif
  - reclass_emp.tif
  - reclass_grund.tif
  - reclass_gym_erhv.tif
  - reclass_lvu.tif
  - reclass_mean_price.tif
  - reclass_mean_sqm.tif
  - reclass_mig_in.tif
  - reclass_mig_net.tif
  - reclass_mig_out.tif
  - reclass_ool.tif
  - reclass_public_housing.tif
  - reclass_qol.tif
  - reclass_unemp.tif

Raster dictionary keys: ['EMUB', 'PMB', 'PUB', 'age_18_25', 'age_26_40', 'age_41_55', 'age_56_69', 'counts', 'crime_main_y', 'disp_inc', 'emp', 'grund', 'gym_erhv', 'lvu', 'mean_price', 'mean_sqm', 'mig_in', 'mig_net', 'mig_out', 'ool', 'public_housing', 'qol', 'unemp']


## Load first raster to get properties

In [16]:
# Load first raster to get CRS and bounds
first_raster_path = list(raster_dict.values())[0]
with rasterio.open(first_raster_path) as src:
    profile = src.profile
    crs = src.crs
    transform = src.transform
    height = src.height
    width = src.width

print(f"CRS: {crs}")
print(f"Dimensions: {width} x {height}")
print(f"Transform: {transform}")

CRS: LOCAL_CS["ETRS89 / UTM zone 32N",UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Dimensions: 128 x 114
Transform: | 100.00, 0.00, 716900.00|
| 0.00,-100.00, 6182000.00|
| 0.00, 0.00, 1.00|


ERROR 1: PROJ: internal_proj_identify: /Users/jacobsmacbookpro/opt/anaconda3/share/proj/proj.db lacks DATABASE.LAYOUT.VERSION.MAJOR / DATABASE.LAYOUT.VERSION.MINOR metadata. It comes from another PROJ installation.


## Step 3: Weighted Overlay

In [17]:
# Check which variables we have in rasters
print(f"Variables in rasters: {len(raster_dict)}")
print(f"Variables in weights: {len(weight_dict)}")

# Find matches between weights and rasters
matched_vars = [v for v in weight_dict.keys() if v in raster_dict]
missing_in_rasters = [v for v in weight_dict.keys() if v not in raster_dict]
extra_in_rasters = [v for v in raster_dict.keys() if v not in weight_dict]

print(f"\n✓ Matched variables: {len(matched_vars)}")
if missing_in_rasters:
    print(f"✗ Missing in rasters: {missing_in_rasters}")
if extra_in_rasters:
    print(f"ℹ Extra in rasters (not in weights): {extra_in_rasters}")

# Use only matched variables for overlay
normalized_weights_matched = {k: v for k, v in normalized_weights.items() if k in raster_dict}

# Re-normalize to sum to 1
total_w = sum(normalized_weights_matched.values())
normalized_weights_matched = {k: v/total_w for k, v in normalized_weights_matched.items()}

print(f"\nNormalized weights (will use {len(normalized_weights_matched)} variables):")
for var, w in sorted(normalized_weights_matched.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {var:30s}: {w:.4f}")
if len(normalized_weights_matched) > 5:
    print(f"  ... and {len(normalized_weights_matched) - 5} more")
print(f"\nSum of normalized weights: {sum(normalized_weights_matched.values()):.6f}")

Variables in rasters: 23
Variables in weights: 20

✓ Matched variables: 20
ℹ Extra in rasters (not in weights): ['counts', 'mig_net', 'qol']

Normalized weights (will use 20 variables):
  disp_inc                      : 0.1205
  public_housing                : 0.0971
  mean_sqm                      : 0.0888
  mean_price                    : 0.0878
  mig_out                       : 0.0711
  ... and 15 more

Sum of normalized weights: 1.000000


In [18]:
# Create weighted overlay
print("Loading rasters and calculating weighted overlay...")
print(f"Using {len(normalized_weights_matched)} weighted variables\n")

overlay_array = np.zeros((height, width), dtype=np.float32)
weight_count = 0

for var_name, weight in sorted(normalized_weights_matched.items(), key=lambda x: x[1], reverse=True):
    if var_name in raster_dict:
        raster_path = raster_dict[var_name]
        print(f"  Loading {var_name:30s} (weight: {weight:.4f})...")
        
        with rasterio.open(raster_path) as src:
            raster_data = src.read(1).astype(np.float32)
            # Mask out nodata/zero values
            raster_data = np.where(raster_data > 0, raster_data, np.nan)
        
        # Add weighted contribution
        overlay_array = np.nansum([overlay_array, raster_data * weight], axis=0)
        weight_count += 1

print(f"\n✓ Weighted overlay created from {weight_count} rasters")
print(f"  Min value: {np.nanmin(overlay_array):.4f}")
print(f"  Max value: {np.nanmax(overlay_array):.4f}")
print(f"  Mean value: {np.nanmean(overlay_array):.4f}")
print(f"  Std value: {np.nanstd(overlay_array):.4f}")

Loading rasters and calculating weighted overlay...
Using 20 weighted variables

  Loading disp_inc                       (weight: 0.1205)...
  Loading public_housing                 (weight: 0.0971)...
  Loading public_housing                 (weight: 0.0971)...
  Loading mean_sqm                       (weight: 0.0888)...
  Loading mean_price                     (weight: 0.0878)...
  Loading mean_sqm                       (weight: 0.0888)...
  Loading mean_price                     (weight: 0.0878)...
  Loading mig_out                        (weight: 0.0711)...
  Loading mig_out                        (weight: 0.0711)...
  Loading unemp                          (weight: 0.0678)...
  Loading lvu                            (weight: 0.0638)...
  Loading unemp                          (weight: 0.0678)...
  Loading lvu                            (weight: 0.0638)...
  Loading emp                            (weight: 0.0616)...
  Loading emp                            (weight: 0.0616)...
  Lo

In [19]:
# Save weighted overlay raster
overlay_path = os.path.join(OUTPUT_FOLDER, "weighted_overlay.tif")

# Delete old file if it exists and is locked
if os.path.exists(overlay_path):
    try:
        os.remove(overlay_path)
        print(f"  Removed old file: {overlay_path}")
    except PermissionError:
        print(f"  WARNING: Could not delete old file (in use). Using backup name...")
        overlay_path = os.path.join(OUTPUT_FOLDER, "weighted_overlay_new.tif")

# Replace NaN with 0 for saving
overlay_array = np.where(np.isnan(overlay_array), 0, overlay_array).astype(np.float32)

try:
    with rasterio.open(
        overlay_path,
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype=np.float32,
        crs=crs,
        transform=transform,
    ) as dst:
        dst.write(overlay_array, 1)

    print(f"✓ Weighted overlay saved: {overlay_path}")
except PermissionError as e:
    print(f"✗ Permission error: {e}")
    print(f"  Try closing the file in ArcGIS/QGIS and running again")
except Exception as e:
    print(f"✗ Error saving raster: {e}")

✓ Weighted overlay saved: /Users/jacobsmacbookpro/P7_pyt/0041overlay_output/weighted_overlay.tif


## Step 5: Load Clusters and Calculate Zonal Statistics

In [20]:
# Load clusters shapefile using fiona directly to avoid geopandas issues
print("Loading clusters shapefile...")

import fiona
from shapely.geometry import shape as geom_from_shape

geometries = []
properties_list = []

with fiona.open(CLUSTERS_SHP) as src:
    crs = src.crs
    for feature in src:
        # Extract geometry and properties
        geom = geom_from_shape(feature['geometry'])
        geometries.append(geom)
        properties_list.append(feature['properties'])

# Create GeoDataFrame manually
clusters_gdf = gpd.GeoDataFrame(
    properties_list,
    geometry=geometries,
    crs=crs
)

print(f"✓ Successfully loaded clusters using fiona")
print(f"\nClusters GeoDataFrame:")
print(clusters_gdf.head())
print(f"\nColumns: {list(clusters_gdf.columns)}")
print(f"Shape: {clusters_gdf.shape}")
print(f"CRS: {clusters_gdf.crs}")

Loading clusters shapefile...
✓ Successfully loaded clusters using fiona

Clusters GeoDataFrame:
   Shape_Area  Shape_Leng cluster_id  fid_1  id_munic munic_clus  munic_code  \
0     70000.0      1400.0      101_1    1.0       1.0      101_1       101.0   
1    110000.0      1800.0      101_2    2.0       2.0      101_2       101.0   
2     30000.0       800.0      101_3    3.0       3.0      101_3       101.0   
3     50000.0      1000.0      101_4    4.0       4.0      101_4       101.0   
4     20000.0       600.0      101_5    5.0       5.0      101_5       101.0   

                                            geometry  
0  POLYGON ((719400 6178100, 719400 6178200, 7195...  
1  POLYGON ((719200 6178100, 719100 6178100, 7191...  
2  POLYGON ((719400 6177700, 719400 6177800, 7194...  
3  POLYGON ((719800 6178200, 719700 6178200, 7196...  
4  POLYGON ((719900 6177800, 719900 6177900, 7200...  

Columns: ['Shape_Area', 'Shape_Leng', 'cluster_id', 'fid_1', 'id_munic', 'munic_clus', 'mun

## Step 6: Perform Zonal Statistics

In [21]:
# Perform Zonal Statistics
print("Calculating zonal statistics for overlay raster...")

from rasterstats import zonal_stats

# Ensure both GeoDataFrames have the same CRS
if clusters_gdf.crs != crs:
    clusters_gdf = clusters_gdf.to_crs(crs)
    print(f"  Reprojected clusters to {crs}")

# Calculate zonal statistics
# stats=['mean', 'count', 'std', 'min', 'max'] calculates these for each zone
stats_list = zonal_stats(
    clusters_gdf.geometry,  # Zones (cluster polygons)
    overlay_path,            # Raster to analyze
    affine=transform,
    stats=['mean', 'count', 'std', 'min', 'max'],
    nodata=0,
    all_touched=False
)

print(f"✓ Calculated zonal statistics for {len(stats_list)} clusters")
print(f"\nSample statistics (first cluster):")
print(stats_list[0])

# Convert to DataFrame and add to clusters GeoDataFrame
stats_df = pd.DataFrame(stats_list)

# Calculate range (max - min)
stats_df['range'] = stats_df['max'] - stats_df['min']

# Merge with clusters GeoDataFrame
result_gdf = clusters_gdf.copy()
for col in stats_df.columns:
    result_gdf[col] = stats_df[col]

print(f"\nResult GeoDataFrame with zonal statistics:")
print(result_gdf[['mean', 'count', 'std', 'range']].head(10))
print(f"\nData types:")
print(result_gdf[['mean', 'count', 'std', 'range']].dtypes)

Calculating zonal statistics for overlay raster...
✓ Calculated zonal statistics for 1421 clusters

Sample statistics (first cluster):
{'min': 3.568476915359497, 'max': 3.568476915359497, 'mean': 3.568477085658482, 'count': 7, 'std': 1.7029898513598596e-07}

Result GeoDataFrame with zonal statistics:
       mean  count           std  range
0  3.568477      7  1.702990e-07    0.0
1  3.555814     11  6.285581e-07    0.0
2  2.947018      3  7.947286e-08    0.0
3  3.260913      5  4.768372e-08    0.0
4  2.443852      2  0.000000e+00    0.0
5  2.068810      4  0.000000e+00    0.0
6  2.459014      4  0.000000e+00    0.0
7  3.216928      5  4.768372e-08    0.0
8  3.000500      9  2.649095e-08    0.0
9  2.767244      3  0.000000e+00    0.0

Data types:
mean     float64
count      int64
std      float64
range    float64
dtype: object
✓ Calculated zonal statistics for 1421 clusters

Sample statistics (first cluster):
{'min': 3.568476915359497, 'max': 3.568476915359497, 'mean': 3.568477085658482,

## Save Results

In [22]:
# Save result as shapefile and GeoJSON
output_shp = os.path.join(OUTPUT_FOLDER, "weighted_overlay_clusters.shp")
result_gdf.to_file(output_shp)
print(f"✓ Result saved as shapefile: {output_shp}")

# Also save as GeoJSON
output_geojson = os.path.join(OUTPUT_FOLDER, "weighted_overlay_clusters.geojson")
result_gdf.to_file(output_geojson, driver='GeoJSON')
print(f"✓ Result saved as GeoJSON: {output_geojson}")

# Also save as CSV for easy inspection
output_csv = os.path.join(OUTPUT_FOLDER, "weighted_overlay_statistics.csv")
result_gdf.drop(columns='geometry').to_csv(output_csv, index=False)
print(f"✓ Statistics saved as CSV: {output_csv}")

# Print summary statistics
print(f"\n{'='*60}")
print(f"ZONAL STATISTICS SUMMARY")
print(f"{'='*60}")
print(f"\nMean values per cluster:")
print(result_gdf['mean'].describe())
print(f"\nCount of pixels per cluster:")
print(result_gdf['count'].describe())
print(f"\nStandard deviation per cluster:")
print(result_gdf['std'].describe())
print(f"\nRange (Max-Min) per cluster:")
print(result_gdf['range'].describe())

✓ Result saved as shapefile: /Users/jacobsmacbookpro/P7_pyt/0041overlay_output/weighted_overlay_clusters.shp
✓ Result saved as GeoJSON: /Users/jacobsmacbookpro/P7_pyt/0041overlay_output/weighted_overlay_clusters.geojson
✓ Statistics saved as CSV: /Users/jacobsmacbookpro/P7_pyt/0041overlay_output/weighted_overlay_statistics.csv

ZONAL STATISTICS SUMMARY

Mean values per cluster:
count    1421.000000
mean        2.699163
std         0.520920
min         1.414028
25%         2.278241
50%         2.677274
75%         3.117961
max         3.971343
Name: mean, dtype: float64

Count of pixels per cluster:
count    1421.000000
mean        4.306122
std         5.467950
min         1.000000
25%         1.000000
50%         2.000000
75%         5.000000
max        48.000000
Name: count, dtype: float64

Standard deviation per cluster:
count    1.421000e+03
mean     3.232296e-08
std      6.606761e-08
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      4.768372e-08
max      6.

In [23]:
import folium
from folium import plugins

# Create base map centered on clusters
center_lat = result_gdf.geometry.centroid.y.mean()
center_lon = result_gdf.geometry.centroid.x.mean()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=11,
    tiles='OpenStreetMap'
)

# Normalize mean values for color mapping
min_val = result_gdf['mean'].min()
max_val = result_gdf['mean'].max()
result_gdf['normalized_mean'] = (result_gdf['mean'] - min_val) / (max_val - min_val) if max_val > min_val else 0

# Define color function (red = low, green = high)
def get_color(value):
    """Return color based on normalized value"""
    if value < 0.2:
        return '#d73027'  # Dark red
    elif value < 0.4:
        return '#fc8d59'  # Orange
    elif value < 0.6:
        return '#fee090'  # Yellow
    elif value < 0.8:
        return '#91bfdb'  # Light blue
    else:
        return '#1a9850'  # Green

# Add clusters to map with colors based on mean overlay value
for idx, row in result_gdf.iterrows():
    norm_val = row['normalized_mean']
    color = get_color(norm_val)
    
    # Create popup with comprehensive statistics
    popup_text = f"""
    <b>Cluster {idx}</b><br>
    <hr>
    <b>Zonal Statistics:</b><br>
    Mean: {row['mean']:.4f}<br>
    Std Dev: {row['std']:.4f}<br>
    Count: {int(row['count'])}<br>
    Range: {row['range']:.4f}<br>
    Min: {row['min']:.4f}<br>
    Max: {row['max']:.4f}
    """
    
    folium.GeoJson(
        data=row.geometry.__geo_interface__,
        style_function=lambda x, color=color: {
            'fillColor': color,
            'color': 'black',
            'weight': 1.5,
            'opacity': 0.9,
            'fillOpacity': 0.7
        },
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(m)

# Add colorbar legend
colorbar_html = '''
<div style="position: fixed; 
     bottom: 50px; right: 50px; width: 220px; height: 300px; 
     background-color: white; border:2px solid grey; z-index:9999; 
     font-size:13px; padding: 10px; border-radius: 5px;">
     <p style="margin:0; font-weight: bold; text-align: center;">Mean Overlay Value</p>
     <hr style="margin: 5px 0;">
     <p style="margin:3px 0;">
     <i style="background: #d73027; width: 18px; height: 18px; display: inline-block; margin-right: 5px; border-radius: 2px;"></i><span style="font-size: 11px;">Very Low (0-20%)</span>
     </p>
     <p style="margin:3px 0;">
     <i style="background: #fc8d59; width: 18px; height: 18px; display: inline-block; margin-right: 5px; border-radius: 2px;"></i><span style="font-size: 11px;">Low (20-40%)</span>
     </p>
     <p style="margin:3px 0;">
     <i style="background: #fee090; width: 18px; height: 18px; display: inline-block; margin-right: 5px; border-radius: 2px;"></i><span style="font-size: 11px;">Medium (40-60%)</span>
     </p>
     <p style="margin:3px 0;">
     <i style="background: #91bfdb; width: 18px; height: 18px; display: inline-block; margin-right: 5px; border-radius: 2px;"></i><span style="font-size: 11px;">High (60-80%)</span>
     </p>
     <p style="margin:3px 0;">
     <i style="background: #1a9850; width: 18px; height: 18px; display: inline-block; margin-right: 5px; border-radius: 2px;"></i><span style="font-size: 11px;">Very High (80-100%)</span>
     </p>
     <hr style="margin: 8px 0;">
     <p style="margin:3px 0; font-size: 11px;"><b>Min:</b> {:.4f}</p>
     <p style="margin:3px 0; font-size: 11px;"><b>Max:</b> {:.4f}</p>
     <p style="margin:3px 0; font-size: 11px;"><b>Clusters:</b> {}</p>
</div>
'''.format(min_val, max_val, len(result_gdf))

m.get_root().html.add_child(folium.Element(colorbar_html))

# Save map
map_path = os.path.join(OUTPUT_FOLDER, "weighted_overlay_zonal_stats_map.html")
m.save(map_path)
print(f"✓ Interactive map saved: {map_path}")

print(f"\nMap Statistics:")
print(f"  Number of clusters: {len(result_gdf)}")
print(f"  Mean value range: {min_val:.4f} to {max_val:.4f}")
print(f"  Overall mean: {result_gdf['mean'].mean():.4f}")
print(f"  Overall std: {result_gdf['std'].mean():.4f}")


✓ Interactive map saved: /Users/jacobsmacbookpro/P7_pyt/0041overlay_output/weighted_overlay_zonal_stats_map.html

Map Statistics:
  Number of clusters: 1421
  Mean value range: 1.4140 to 3.9713
  Overall mean: 2.6992
  Overall std: 0.0000
